In [6]:
import pandas as pd
import numpy as np
from numba import njit

In [7]:
sept = pd.read_pickle("./pickles/sept.pickle")
sept

,text,str,verse,chapter,book,rmac
0,εν,1722,1,1,1,prep
1,αρχη,746,1,1,1,n-dsf
2,εποιησεν,4160,1,1,1,v-aai-3s
3,ο,3588,1,1,1,t-nsm
4,θεος,2316,1,1,1,n-nsm
...,...,...,...,...,...,...
475395,και,2532,6,4,39,conj
475396,παταξω,3960,6,4,39,v-fai-1s
475397,την,3588,6,4,39,t-asf
475398,γην,1065,6,4,39,n-asf


In [8]:
nt = pd.read_pickle("./pickles/tisch.pickle")  # from Isaac's PS1
nt

,text,str,verse,chapter,book,rmac
0,βίβλος,976,1,1,40,n-nsf
1,γενέσεως,1078,1,1,40,n-gsf
2,ἰησοῦ,2424,1,1,40,n-gsm
3,χριστοῦ,5547,1,1,40,n-gsm
4,υἱοῦ,5207,1,1,40,n-gsm
...,...,...,...,...,...,...
137497,τοῦ,3588,21,22,66,t-gsm
137498,κυρίου,2962,21,22,66,n-gsm
137499,ἰησοῦ,2424,21,22,66,n-gsm
137500,μετὰ,3326,21,22,66,prep


In [9]:
@njit(parallel=True)
def _find_matching_ngrams(left, right, min_n=5):
    """
    Helper function for `find_matching_ngrams`
    This does the actual computation, operating on numpy arrays
    """
    found_values = []
    lidx = 0
    while lidx < len(left):
        for ridx in range(len(right)):
            length = 0
            while right[ridx + length] == left[lidx + length]:
                length += 1
            if length >= min_n:
                found_values.append((lidx, ridx, length))
                lidx += length - 1
        lidx += 1
    found = np.array(found_values)
    return found[:, 0], found[:, 1], found[:, 2]


def find_matching_ngrams(left, right, min_n=5):
    """
    Itererate `n` from `max_n` to `min_n` (inclusive). For each n-gram in `left`, look for matching occurances in `right`
    """
    left_ids, right_ids, lengths = _find_matching_ngrams(
        np.array(left), np.array(right), min_n
    )
    return pd.DataFrame(
        {
            "left_id": left_ids,
            "left_len": lengths,
            "right_id": right_ids,
            "right_len": lengths,
        }
    )

In [13]:
sept["str"].astype(str)

0          1722
1           746
2          4160
3          3588
4          2316
          ...  
475395     2532
475396     3960
475397     3588
475398     1065
475399    C6079
Name: str, Length: 475400, dtype: object

In [14]:
find_matching_ngrams(sept["str"].astype(str), nt["str"].astype(str))

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
non-precise type array(pyobject, 1d, C)
During: typing of argument at /tmp/ipykernel_9011/566107697.py (1)

File "../../../../../tmp/ipykernel_9011/566107697.py", line 1:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference